In [ ]:

# IO benchmark: read throughput of the mounted Flash-Next model vs. reader threads.
# Each setting reads distinct, not-yet-read shard files (no page-cache reuse).
import os, time, glob, json
from concurrent.futures import ThreadPoolExecutor
MODEL_DIR = '/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'
files = sorted(glob.glob(MODEL_DIR + '/**/*.safetensors', recursive=True), key=os.path.getsize, reverse=True)
print(len(files), 'files', sum(map(os.path.getsize, files))/1e9, 'GB')
CH = 32 * 1024 * 1024
def read_chunk(a):
    p, off = a
    with open(p, 'rb', buffering=0) as f:
        f.seek(off); n = 0
        while n < CH:
            b = f.read(min(8*1024*1024, CH - n))
            if not b: break
            n += len(b)
    return n
res = []
it = iter(files)
for threads in [1, 3, 8, 16, 32, 64]:
    fs = [next(it), next(it)]
    chunks = [(p, o) for p in fs for o in range(0, os.path.getsize(p), CH)]
    t0 = time.time()
    with ThreadPoolExecutor(threads) as ex:
        tot = sum(ex.map(read_chunk, chunks))
    dt = time.time() - t0
    r = dict(threads=threads, gb=tot/1e9, sec=round(dt,1), mbps=round(tot/1e6/dt,1))
    print('IOBENCH', json.dumps(r), flush=True); res.append(r)
json.dump(res, open('/kaggle/working/io_bench.json','w'))
import pandas as pd
pd.DataFrame([['1_0','1',True,1]], columns=['row_id','game_id','end_of_game','score']).to_parquet('/kaggle/working/submission.parquet', index=False)
